# Synthetic Data Generator
### Step 7 : Kafka Producer Adapter

In [ ]:
import os

from utils.database.postgres import (
    get_engine_from_env, 
    read_sql_dataframe,
)

from utils.synthetic.pipeline.kafka_producer_adapter import (
    run_send_queue_producer_once, 
    run_send_queue_producer_loop, 
    build_sensor_message_payload, 
    json_dumps_safe,
)

from utils.core.env_helpers import (
    env_float,
    env_int,
    env_optional_int,
    env_str,
)

In [ ]:
SCHEMA = env_str("CAPSTONE_SCHEMA", "capstone")

DATASET_ID = env_str(
    "SYNTHETIC_DATASET_ID",
    "pump_synthetic_v1",
    aliases=("DATASET_ID",),
)

RUN_ID = env_str(
    "SYNTHETIC_RUN_ID",
    "synthetic_run_001",
    aliases=("RUN_ID",),
)

SIMULATION_TABLE_NAME = env_str(
    "SYNTHETIC_CONTROL_TABLE",
    "simulation_state_control",
    aliases=("PRODUCER_CONTROL_TABLE",),
)

SEND_QUEUE_TABLE_NAME = env_str(
    "SYNTHETIC_SEND_QUEUE_TABLE",
    "synthetic_sensor_messages_send_queue",
    aliases=("PRODUCER_QUEUE_TABLE",),
)

PRODUCER_TOPIC = env_str(
    "SYNTHETIC_KAFKA_TOPIC",
    "pump.telemetry.synthetic",
    aliases=("KAFKA_TOPIC",),
)

PRODUCER_WORKER_ID = env_str(
    "SYNTHETIC_PRODUCER_WORKER_ID",
    "producer_worker_test_001",
)

CLIENT_ID = env_str(
    "SYNTHETIC_PRODUCER_CLIENT_ID",
    "synthetic-telemetry-producer",
    aliases=("KAFKA_CLIENT_ID", "SYNTHETIC_PRODUCER_GROUP_ID"),
)

OBSERVATION_BATCH_SIZE = env_int("OBSERVATION_BATCH_SIZE", 500)
PRODUCER_POLL_SECONDS = env_float("PRODUCER_POLL_SECONDS", 0.0)
PRODUCER_MAX_SEND_ATTEMPTS = env_int("PRODUCER_MAX_SEND_ATTEMPTS", 3)
FLUSH_TIMEOUT_SECONDS = env_float("PRODUCER_FLUSH_TIMEOUT_SECONDS", 30.0)

RUN_PRODUCER_LOOP_FLAG = True
RUN_ONE_BATCH_PRODUCER_SMOKE_TEST_FLAG = False

MAX_BATCHES = env_optional_int(
    "PRODUCER_MAX_BATCHES_LIMIT",
    default=1,
    aliases=("SYNTHETIC_PRODUCER_MAX_BATCHES_LIMIT",),
)

STOP_ON_FAILURE_FLAG = True

print("Producer Adapter config")
print("schema:", SCHEMA)
print("dataset id:", DATASET_ID)
print("run id:", RUN_ID)
print("send queue table:", SEND_QUEUE_TABLE_NAME)
print("producer topic:", PRODUCER_TOPIC)
print("observation batch size:", OBSERVATION_BATCH_SIZE)
print("client id:", CLIENT_ID)
print("flush timeout seconds:", FLUSH_TIMEOUT_SECONDS)
print("run producer loop:", RUN_PRODUCER_LOOP_FLAG)
print("producer max batches:", MAX_BATCHES)
print("stop on failure:", STOP_ON_FAILURE_FLAG)



Producer Adapter config
schema: capstone
dataset id: pump_synthetic_v1
run id: synthetic_run_001
send queue table: synthetic_sensor_messages_send_queue
producer topic: pump.telemetry.synthetic
observation batch size: 500
client id: pump-telemetry-producer
flush timeout seconds: 30.0
run producer loop: True
producer max batches limit raw: None
producer max batches: None
stop on failure: True


---

In [14]:
engine = get_engine_from_env()

---

In [5]:
NUMBER_OF_SENSORS = int(
    read_sql_dataframe(
        engine,
        f"""
        SELECT COUNT(DISTINCT sensor_index) AS sensor_count
        FROM "{SCHEMA}"."{SEND_QUEUE_TABLE_NAME}"
        WHERE dataset_id = :dataset_id
          AND run_id = :run_id
        """,
        params={
            "dataset_id": DATASET_ID,
            "run_id": RUN_ID,
        },
    ).loc[0, "sensor_count"]
)

PRODUCER_BATCH_SIZE = OBSERVATION_BATCH_SIZE * NUMBER_OF_SENSORS

print("Derived producer batch sizing")
print("number of sensors:", NUMBER_OF_SENSORS)
print("observation batch size:", OBSERVATION_BATCH_SIZE)
print("producer batch size:", PRODUCER_BATCH_SIZE)

Derived producer batch sizing
number of sensors: 52
observation batch size: 500
producer batch size: 26000


----

In [6]:
pre_send_queue_status_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        queue_status,
        COUNT(*) AS row_count,
        COUNT(*) FILTER (WHERE claim_token IS NULL) AS null_claim_token_count,
        COUNT(*) FILTER (WHERE claim_token IS NOT NULL) AS populated_claim_token_count,
        COUNT(*) FILTER (WHERE producer_sent_at IS NULL) AS unsent_count,
        COUNT(*) FILTER (WHERE producer_sent_at IS NOT NULL) AS sent_count
    FROM "{SCHEMA}"."{SEND_QUEUE_TABLE_NAME}"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    GROUP BY queue_status
    ORDER BY row_count DESC
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(pre_send_queue_status_dataframe)

,queue_status,row_count,null_claim_token_count,populated_claim_token_count,unsent_count,sent_count
0,pending,11674000,11674000,0,11674000,0
1,sent,26000,0,26000,0,26000


----

In [7]:
preview_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT *
    FROM "{SCHEMA}"."{SEND_QUEUE_TABLE_NAME}"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
      AND queue_status = 'pending'
    ORDER BY observation_index, message_sequence_index, sensor_index
    LIMIT 1
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

if preview_dataframe.empty:
    print("No pending rows available for payload preview.")
else:
    payload = build_sensor_message_payload(preview_dataframe.iloc[0].to_dict())
    print(json_dumps_safe(payload))

{"message_key": "pump_synthetic_v1|synthetic_run_001|pump_asset_001|501|0", "dataset_id": "pump_synthetic_v1", "run_id": "synthetic_run_001", "asset_id": "pump_asset_001", "generated_row_id": "synthetic_run_001_obs_000000000501", "observation": {"observation_index": 501, "observation_timestamp": "2026-04-16T08:20:00+00:00", "batch_id": 1, "row_in_batch": 500, "global_cycle_id": 501, "stream_state": "normal", "phase": "normal"}, "sensor": {"sensor_name": "sensor_00", "sensor_index": 0, "sensor_value": 2.4047780789116113, "message_sequence_index": 0}, "metadata": {"meta_episode_id": 0, "meta_primary_fault_type": "drift_down", "meta_magnitude": 1.2124996276063957, "created_at": "2026-05-24T23:54:01.146754+00:00"}, "telemetry": {"is_telemetry_event": false, "telemetry_event_type": null}, "producer": {"producer_send_attempt": 1, "queued_at": "2026-05-26T00:29:51.868662+00:00"}}


----

#### One Batch

In [8]:
# -----------------------------------------------------------------------------
# Optional smoke test: run exactly one producer batch
# -----------------------------------------------------------------------------
# This is not the normal official Stage 7 execution path.
# Use only when you want to test one batch before running the loop.

#RUN_ONE_BATCH_PRODUCER_SMOKE_TEST_FLAG = False

if RUN_ONE_BATCH_PRODUCER_SMOKE_TEST_FLAG:
    one_batch_result = run_send_queue_producer_once(
        engine=engine,
        dataset_id=DATASET_ID,
        run_id=RUN_ID,
        schema=SCHEMA,
        queue_table=SEND_QUEUE_TABLE_NAME,
        control_table=SIMULATION_TABLE_NAME,
        producer_worker_id=PRODUCER_WORKER_ID,
        client_id=CLIENT_ID,
        flush_timeout_seconds=FLUSH_TIMEOUT_SECONDS,
    )

    display(one_batch_result)
else:
    print("RUN_ONE_BATCH_PRODUCER_SMOKE_TEST_FLAG is False. Skipping one-batch smoke test.")

RUN_ONE_BATCH_PRODUCER_SMOKE_TEST_FLAG is False. Skipping one-batch smoke test.


----

In [9]:
post_send_queue_status_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        queue_status,
        COUNT(*) AS row_count,
        COUNT(*) FILTER (WHERE claim_token IS NULL) AS null_claim_token_count,
        COUNT(*) FILTER (WHERE claim_token IS NOT NULL) AS populated_claim_token_count,
        COUNT(*) FILTER (WHERE producer_sent_at IS NULL) AS unsent_count,
        COUNT(*) FILTER (WHERE producer_sent_at IS NOT NULL) AS sent_count,
        COUNT(*) FILTER (WHERE producer_delivery_error IS NOT NULL) AS error_count
    FROM "{SCHEMA}"."{SEND_QUEUE_TABLE_NAME}"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    GROUP BY queue_status
    ORDER BY row_count DESC
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(post_send_queue_status_dataframe)

,queue_status,row_count,null_claim_token_count,populated_claim_token_count,unsent_count,sent_count,error_count
0,pending,11674000,11674000,0,11674000,0,0
1,sent,26000,0,26000,0,26000,0


----

#### Loop

In [10]:
# -----------------------------------------------------------------------------
# Run Kafka producer loop
# -----------------------------------------------------------------------------
# Official Stage 7 execution path.
#
# For first test:
#   RUN_PRODUCER_LOOP_FLAG = True
#   MAX_BATCHES = 1
#
# For full queue drain:
#   RUN_PRODUCER_LOOP_FLAG = True
#   MAX_BATCHES = None
#
# The loop stops when:
# - queue is empty
# - control row disables the run
# - max_batches is reached
# - a failure occurs and stop_on_failure=True

#RUN_PRODUCER_LOOP_FLAG = False

if RUN_PRODUCER_LOOP_FLAG:
    loop_results = run_send_queue_producer_loop(
        engine=engine,
        dataset_id=DATASET_ID,
        run_id=RUN_ID,
        schema=SCHEMA,
        queue_table=SEND_QUEUE_TABLE_NAME,
        control_table=SIMULATION_TABLE_NAME,
        producer_worker_id=PRODUCER_WORKER_ID,
        client_id=CLIENT_ID,
        max_batches=MAX_BATCHES,
        stop_on_failure=STOP_ON_FAILURE_FLAG,
        flush_timeout_seconds=FLUSH_TIMEOUT_SECONDS,
        enable_progress_logging=True,
        progress_every_batches=1,
    )

    display(loop_results)
else:
    print("RUN_PRODUCER_LOOP_FLAG is False. Skipping producer loop.")

[{'status': 'sent',
  'claimed_rows': 26000,
  'sent_rows': 26000,
  'failed_rows': 0,
  'topic': 'pump.telemetry.synthetic',
  'claim_token': '0cec68cd-8bb1-4dbe-8bda-546386c0084b',
  'errors': [],
  'delivered_count': 26000,
  'expected_count': 26000},
 {'status': 'sent',
  'claimed_rows': 26000,
  'sent_rows': 26000,
  'failed_rows': 0,
  'topic': 'pump.telemetry.synthetic',
  'claim_token': 'b6f782cc-3f93-4347-b080-7fe9f9e3fa36',
  'errors': [],
  'delivered_count': 26000,
  'expected_count': 26000},
 {'status': 'sent',
  'claimed_rows': 26000,
  'sent_rows': 26000,
  'failed_rows': 0,
  'topic': 'pump.telemetry.synthetic',
  'claim_token': '975380f3-70c6-4ace-9da1-37f044c00c79',
  'errors': [],
  'delivered_count': 26000,
  'expected_count': 26000},
 {'status': 'sent',
  'claimed_rows': 26000,
  'sent_rows': 26000,
  'failed_rows': 0,
  'topic': 'pump.telemetry.synthetic',
  'claim_token': '33c11648-5220-4965-a15a-97dd82016196',
  'errors': [],
  'delivered_count': 26000,
  'expec

---

In [13]:
stage_7_final_validation_dataframe = read_sql_dataframe(
    engine,
    f"""
    WITH queue_summary AS (
        SELECT
            COUNT(*) AS total_queue_rows,
            COUNT(*) FILTER (WHERE queue_status = 'sent') AS sent_count,
            COUNT(*) FILTER (WHERE queue_status = 'pending') AS pending_count,
            COUNT(*) FILTER (WHERE queue_status = 'claimed') AS claimed_count,
            COUNT(*) FILTER (WHERE queue_status = 'failed') AS failed_count,
            COUNT(*) FILTER (WHERE producer_sent_at IS NOT NULL) AS sent_timestamp_count,
            COUNT(*) FILTER (WHERE claim_token IS NOT NULL) AS populated_claim_token_count,
            COUNT(*) FILTER (WHERE producer_delivery_error IS NOT NULL) AS error_count,
            COUNT(DISTINCT message_key) AS distinct_message_key_count,
            COUNT(DISTINCT observation_index) AS distinct_observation_count
        FROM "{SCHEMA}"."{SEND_QUEUE_TABLE_NAME}"
        WHERE dataset_id = :dataset_id
          AND run_id = :run_id
    )
    SELECT
        *,
        (
            total_queue_rows > 0
            AND sent_count = total_queue_rows
            AND pending_count = 0
            AND claimed_count = 0
            AND failed_count = 0
            AND sent_timestamp_count = total_queue_rows
            AND error_count = 0
        ) AS ready_for_stage_8
    FROM queue_summary
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(stage_7_final_validation_dataframe)

,total_queue_rows,sent_count,pending_count,claimed_count,failed_count,sent_timestamp_count,populated_claim_token_count,error_count,distinct_message_key_count,distinct_observation_count,ready_for_stage_8
0,11700000,11700000,0,0,0,11700000,11700000,0,11700000,225000,True


---

In [11]:
# Dispose Engine
engine.dispose()

In [17]:
consumer_progress_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        COUNT(*) AS consumed_row_count,
        COUNT(DISTINCT message_key) AS distinct_message_key_count,
        COUNT(DISTINCT observation_index) AS distinct_observation_count,
        COUNT(*) FILTER (WHERE rebuild_status = 'pending') AS pending_rebuild_count,
        MIN(consumer_received_at) AS first_consumed_at,
        MAX(consumer_received_at) AS last_consumed_at
    FROM "{SCHEMA}"."synthetic_sensor_messages_consumed_stage"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(consumer_progress_dataframe)

,consumed_row_count,distinct_message_key_count,distinct_observation_count,pending_rebuild_count,first_consumed_at,last_consumed_at
0,930800,930800,18119,930800,2026-05-26 22:48:33.489191+00:00,2026-05-26 22:59:37.612822+00:00


In [18]:
consumed_global_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        COUNT(*) AS total_consumed_rows,
        COUNT(DISTINCT dataset_id) AS distinct_dataset_count,
        COUNT(DISTINCT run_id) AS distinct_run_count,
        COUNT(DISTINCT message_key) AS distinct_message_key_count,
        COUNT(DISTINCT observation_index) AS distinct_observation_count,
        MIN(consumer_received_at) AS first_consumed_at,
        MAX(consumer_received_at) AS last_consumed_at
    FROM "{SCHEMA}"."synthetic_sensor_messages_consumed_stage"
    """
)

display(consumed_global_dataframe)

,total_consumed_rows,distinct_dataset_count,distinct_run_count,distinct_message_key_count,distinct_observation_count,first_consumed_at,last_consumed_at
0,8632000,1,1,8632000,166171,2026-05-26 22:48:33.489191+00:00,2026-05-26 23:58:17.986617+00:00


In [19]:
consumed_dataset_run_breakdown_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        dataset_id,
        run_id,
        consumer_group_id,
        consumer_worker_id,
        COUNT(*) AS row_count,
        COUNT(DISTINCT message_key) AS distinct_message_key_count,
        COUNT(DISTINCT observation_index) AS distinct_observation_count,
        MIN(consumer_received_at) AS first_consumed_at,
        MAX(consumer_received_at) AS last_consumed_at
    FROM "{SCHEMA}"."synthetic_sensor_messages_consumed_stage"
    GROUP BY dataset_id, run_id, consumer_group_id, consumer_worker_id
    ORDER BY row_count DESC
    LIMIT 25
    """
)

display(consumed_dataset_run_breakdown_dataframe)

,dataset_id,run_id,consumer_group_id,consumer_worker_id,row_count,distinct_message_key_count,distinct_observation_count,first_consumed_at,last_consumed_at
0,pump_synthetic_v1,synthetic_run_001,synthetic-telemetry-consumer-group-run-001,4f3a29e849ee,4440800,4440800,176684,2026-05-26 22:48:33.489191+00:00,2026-05-27 00:03:12.633389+00:00
1,pump_synthetic_v1,synthetic_run_001,synthetic-telemetry-consumer-group-run-001,consumer_worker_002,2371200,2371200,136263,2026-05-26 23:13:31.908792+00:00,2026-05-27 00:03:08.683108+00:00
2,pump_synthetic_v1,synthetic_run_001,synthetic-telemetry-consumer-group-run-001,consumer_worker_003,2360800,2360800,136152,2026-05-26 23:13:45.174453+00:00,2026-05-27 00:03:12.662184+00:00


In [ ]:
stage_8_global_validation_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        COUNT(*) AS total_consumed_rows,
        COUNT(DISTINCT dataset_id) AS distinct_dataset_count,
        COUNT(DISTINCT run_id) AS distinct_run_count,
        COUNT(DISTINCT message_key) AS distinct_message_key_count,
        COUNT(DISTINCT observation_index) AS distinct_observation_count,
        COUNT(*) FILTER (WHERE rebuild_status = 'pending') AS pending_rebuild_count,
        COUNT(*) FILTER (WHERE message_key IS NULL) AS null_message_key_count,
        COUNT(*) FILTER (WHERE sensor_value IS NULL) AS null_sensor_value_count,
        MIN(consumer_received_at) AS first_consumed_at,
        MAX(consumer_received_at) AS last_consumed_at
    FROM "{SCHEMA}"."synthetic_sensor_messages_consumed_stage"
    """
)

display(stage_8_global_validation_dataframe)

In [ ]:
stage_8_final_validation_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        COUNT(*) AS consumed_row_count,
        COUNT(DISTINCT message_key) AS distinct_message_key_count,
        COUNT(DISTINCT observation_index) AS distinct_observation_count,
        COUNT(DISTINCT sensor_name) AS distinct_sensor_name_count,
        COUNT(DISTINCT sensor_index) AS distinct_sensor_index_count,
        COUNT(*) FILTER (WHERE rebuild_status = 'pending') AS pending_rebuild_count,
        COUNT(*) FILTER (WHERE is_duplicate = TRUE) AS duplicate_offset_count,
        COUNT(*) FILTER (WHERE message_key IS NULL) AS null_message_key_count,
        COUNT(*) FILTER (WHERE sensor_value IS NULL) AS null_sensor_value_count,
        MIN(observation_index) AS min_observation_index,
        MAX(observation_index) AS max_observation_index,
        MIN(consumer_received_at) AS first_consumed_at,
        MAX(consumer_received_at) AS last_consumed_at,
        (
            COUNT(*) = 11700000
            AND COUNT(DISTINCT message_key) = 11700000
            AND COUNT(DISTINCT observation_index) = 225000
            AND COUNT(DISTINCT sensor_index) = 52
            AND COUNT(*) FILTER (WHERE rebuild_status = 'pending') = 11700000
            AND COUNT(*) FILTER (WHERE message_key IS NULL) = 0
            AND COUNT(*) FILTER (WHERE sensor_value IS NULL) = 0
        ) AS ready_for_stage_9
    FROM "{SCHEMA}"."synthetic_sensor_messages_consumed_stage"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(stage_8_final_validation_dataframe)